
This Databricks notebook, titled **understanding_quick_comm**, is designed to perform a comprehensive end-to-end data engineering and analytics workflow on an e-commerce dataset (Olist).

The notebook is organized into several key phases:

- **Data Ingestion & Integration:** It loads multiple CSV files (orders, items, payments, reviews, etc.) using pandas and merges them into a single Master Dataset (df) for unified analysis.
- **Data Transformation:** It handles schema cleaning, such as converting timestamp strings into datetime objects and translating product category names from Portuguese to English.
- **Feature Engineering:** It calculates business-specific metrics, such as delivery_delay_days, and categorizes orders as "Delayed" or "On Time."
- **Data Persistence:** It converts the processed Pandas DataFrame into a Spark DataFrame to save it as a permanent Delta table (final_quick_comm_dataset) in the Databricks workspace catalog.
- **Business Intelligence:** The final sections generate high-level Executive KPIs (GMV, AOV, Repeat Purchase Rate) and analyze the correlation between delivery performance and customer satisfaction scores.

# Load the Files First in DataFrames

In [0]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv("quick_comm_tables/olist_orders_dataset.csv")
items = pd.read_csv("quick_comm_tables/olist_order_items_dataset.csv")
payments = pd.read_csv("quick_comm_tables/olist_order_payments_dataset.csv")
reviews = pd.read_csv("quick_comm_tables/olist_order_reviews_dataset.csv")
products = pd.read_csv("quick_comm_tables/olist_products_dataset.csv")
customers = pd.read_csv("quick_comm_tables/olist_customers_dataset.csv")
category = pd.read_csv("quick_comm_tables/product_category_name_translation.csv")

## Translate Product Categories

In [0]:
products = products.merge(
    category,
    on="product_category_name",
    how="left"
)

## Build MASTER DATASET

In [0]:
df = orders.merge(items, on="order_id", how="left") \
           .merge(payments, on="order_id", how="left") \
           .merge(reviews, on="order_id", how="left") \
           .merge(products, on="product_id", how="left") \
           .merge(customers, on="customer_id", how="left")

In [0]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

In [0]:
%sql
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 100

In [0]:
df.describe()

In [0]:
df.isnull().sum()

## Create BUSINESS METRICS

In [0]:
#delivery delay
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] -
    df["order_estimated_delivery_date"]
).dt.days
#delivery status
df["delivery_status"] = np.where(
    df["delivery_delay_days"] > 0,
    "Delayed",
    "On Time"
)

# CREATE dataset with the above df

In [0]:
%sql
SELECT current_catalog() as catalog, current_schema() as schema

In [0]:
# Save as a permanent Delta table with explicit catalog and schema
spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.final_quick_comm_dataset")

In [0]:
%sql
SELECT * FROM workspace.default.final_quick_comm_dataset LIMIT 100

# Generate Executive KPIs

In [0]:
total_orders = df["order_id"].nunique()
gmv = df["payment_value"].sum()
aov = gmv / total_orders
avg_review = df["review_score"].mean()

print("Total Orders:", total_orders)
print("GMV:", round(gmv,2))
print("AOV:", round(aov,2))
print("Avg Review:", round(avg_review,2))

# Analysis 1 — Delivery Delay vs Customer Satisfaction

In [0]:
df.groupby("delivery_status")["review_score"].mean()

# Analysis 2 — Top Revenue Categories

In [0]:
category_perf = df.groupby(
    "product_category_name_english"
).agg({
    "payment_value":"sum",
    "review_score":"mean",
    "order_id":"nunique"
}).sort_values(
    by="payment_value",
    ascending=False
).head(10)

category_perf

# Analysis 3 — Repeat Customer Analysis

In [0]:
customer_orders = df.groupby(
    "customer_unique_id"
)["order_id"].nunique().reset_index()

customer_orders.columns = [
    "customer_unique_id",
    "order_count"
]

In [0]:
repeat_rate = (
    customer_orders["order_count"] > 1
).mean()

print("Repeat Purchase Rate:", round(repeat_rate*100,2), "%")